# Requirements


In [ ]:
!pip install -q -U --force-reinstall langchain==0.2.17 langchain-community langchain-core langchain-text-splitters langchain-google-genai chromadb pypdf tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 10.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# API key

In [ ]:
import os
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")
print("API key set.")

In [ ]:
from google.colab import files
files.upload()

{}

# Data or PFD for RAG

In [ ]:
PDF_PATHS = [
    "/content/1_Constitutional_Amendments.pdf",
    "/content/2_Family_and_Civil_Laws.pdf",
    "/content/3_Criminal_Business_Rights_Laws.pdf",
    "/content/4_Key_Constitutional_Articles.pdf",
    "/content/5_Constitution_of_Pakistan.pdf",
]

for p in PDF_PATHS:
    print(p, "-> exists:", os.path.exists(p))

/content/1_Constitutional_Amendments.pdf -> exists: True
/content/2_Family_and_Civil_Laws.pdf -> exists: True
/content/3_Criminal_Business_Rights_Laws.pdf -> exists: True
/content/4_Key_Constitutional_Articles.pdf -> exists: True
/content/5_Constitution_of_Pakistan.pdf -> exists: True


In [ ]:
#!pip uninstall -y langchain
#!pip install langchain==0.2.17

# Libraries and embedding model

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
import time
from tqdm.auto import tqdm

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ModuleNotFoundError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

try:
    from langchain_core.prompts import PromptTemplate
except ModuleNotFoundError:
    from langchain.prompts import PromptTemplate

persist_directory = "./chroma_db"

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

In [ ]:
def load_chunk_embed_store(path, vectorstore, embeddings, batch_size=20, delay=2, max_retries=5):
    print(f"Loading: {path}")
    loader = PyPDFLoader(path)
    docs = loader.load()

    chunks = text_splitter.split_documents(docs)
    print(f"  {len(docs)} pages -> {len(chunks)} chunks")

    for i in tqdm(range(0, len(chunks), batch_size), desc=f"Embedding {os.path.basename(path)}"):
        batch = chunks[i:i + batch_size]

        for attempt in range(max_retries):
            try:
                if vectorstore is None:
                    vectorstore = Chroma.from_documents(
                        documents=batch,
                        embedding=embeddings,
                        persist_directory=persist_directory
                    )
                else:
                    vectorstore.add_documents(batch)
                break
            except Exception as e:
                wait = delay * (2 ** attempt)
                print(f"  Batch failed ({e}), retrying in {wait}s...")
                time.sleep(wait)
        else:
            print(f"  Batch at index {i} failed after {max_retries} retries — skipping")
            continue

        time.sleep(delay)

    print(f"Done: {path}\n")
    return vectorstore

In [ ]:
vectorstore = None

for path in PDF_PATHS:
    if not os.path.exists(path):
        print(f"Skipping missing file: {path}")
        continue
    vectorstore = load_chunk_embed_store(path, vectorstore, embeddings)

print("All PDFs processed. Vector store ready at:", persist_directory)

Loading: /content/1_Constitutional_Amendments.pdf
  6 pages -> 13 chunks


Embedding 1_Constitutional_Amendments.pdf:   0%|          | 0/1 [00:00<?, ?it/s]

Done: /content/1_Constitutional_Amendments.pdf

Loading: /content/2_Family_and_Civil_Laws.pdf
  7 pages -> 14 chunks


Embedding 2_Family_and_Civil_Laws.pdf:   0%|          | 0/1 [00:00<?, ?it/s]

Done: /content/2_Family_and_Civil_Laws.pdf

Loading: /content/3_Criminal_Business_Rights_Laws.pdf
  9 pages -> 15 chunks


...

Done: /content/3_Criminal_Business_Rights_Laws.pdf

Loading: /content/4_Key_Constitutional_Articles.pdf
  6 pages -> 9 chunks


Embedding 4_Key_Constitutional_Articles.pdf:   0%|          | 0/1 [00:00<?, ?it/s]

Done: /content/4_Key_Constitutional_Articles.pdf

Loading: /content/5_Constitution_of_Pakistan.pdf
  5 pages -> 10 chunks


Embedding 5_Constitution_of_Pakistan.pdf:   0%|          | 0/1 [00:00<?, ?it/s]

Done: /content/5_Constitution_of_Pakistan.pdf

All PDFs processed. Vector store ready at: ./chroma_db


# Defining the prompt to LLM

In [ ]:
LEGAL_SYSTEM_PROMPT = """You are an expert legal advisor and a professional lawyer.
Answer the user's question to the point ONLY using the context provided below, which comes from the user's own
legal reference documents. Be precise, cite the relevant Article/Section numbers when they appear in
the context, and explain the reasoning clearly in one line maximum , the way an experienced lawyer would advise a client.

If the answer is not contained in the provided context, say so clearly instead of guessing — do not
invent laws, sections, or case outcomes.

Context:
{context}

Question:
{question}

Answer as the expert legal advisor:"""

prompt = PromptTemplate(
    template=LEGAL_SYSTEM_PROMPT,
    input_variables=["context", "question"]
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True,
)

print("Legal advisor chain ready.")

Legal advisor chain ready.


# Checking or prediction

In [ ]:
def ask(question):
    result = qa_chain.invoke({"query": question})
    print("ANSWER:\n")
    print(result["result"])
    print("\n" + "-"*60)
    print("SOURCES:")
    for doc in result["source_documents"]:
        src = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page", "?")
        print(f"  - {os.path.basename(src)}, page {page}")

ask("dost ka pen chori krny per")

ANSWER:

Under Section 379 of the Pakistan Penal Code (PPC), stealing your friend's pen constitutes the offence of theft, which is punishable by imprisonment of up to 3 years, a fine, or both.

------------------------------------------------------------
SOURCES:
  - 3_Criminal_Business_Rights_Laws.pdf, page 1
  - 3_Criminal_Business_Rights_Laws.pdf, page 2
  - 3_Criminal_Business_Rights_Laws.pdf, page 2
  - 3_Criminal_Business_Rights_Laws.pdf, page 3
  - 3_Criminal_Business_Rights_Laws.pdf, page 0


# Gradio Interface

In [ ]:
# ============================================================
#  Pakistan Constitution AI Assistant — Gradio UI (Colab)
#  Fix: theme tokens forced for BOTH light & dark variants
#  (kills the faded text / black textbox / black page-edge bugs)
#  Paste AFTER your RAG cells (needs `qa_chain` to exist)
# ============================================================
!pip install -q gradio

import gradio as gr
import os

# ---------------- Backend hook into your existing RAG chain ----------------
def run_rag(question, history):
    if not question or not question.strip():
        return history, "", gr.update(visible=len(history) == 0), gr.update(visible=len(history) > 0)
    try:
        result = qa_chain.invoke({"query": question})
        answer = result["result"]
        sources = result.get("source_documents", [])
        if sources:
            seen, src_lines = set(), []
            for doc in sources:
                src = os.path.basename(doc.metadata.get("source", "unknown"))
                page = doc.metadata.get("page", "?")
                key = (src, page)
                if key not in seen:
                    seen.add(key)
                    src_lines.append(f"- {src}, page {page}")
            answer += "\n\n**Sources:**\n" + "\n".join(src_lines)
    except Exception as e:
        answer = f"⚠️ Error while retrieving answer: {e}"

    history = history + [{"role": "user", "content": question},
                          {"role": "assistant", "content": answer}]
    return history, "", gr.update(visible=False), gr.update(visible=True)

# ---------------- Colors ----------------
DEEP_GREEN = "#0d3d2b"
MID_GREEN  = "#1a5c3f"
GOLD       = "#b8935a"
CREAM      = "#f7f3ea"
CARD_BORDER = "#e2dcc8"
TEXT_DARK  = "#1a2e22"
TEXT_GRAY  = "#6b6b5f"
USER_BUBBLE = "#e4efe7"
BOT_BUBBLE  = "#faf8f2"

# ---------------- Theme: force identical look in light AND dark mode ----------------
theme = gr.themes.Soft(
    primary_hue=gr.themes.colors.green,
    neutral_hue=gr.themes.colors.stone,
).set(
    # page
    body_background_fill=CREAM, body_background_fill_dark=CREAM,
    body_text_color=TEXT_DARK, body_text_color_dark=TEXT_DARK,
    body_text_color_subdued=TEXT_GRAY, body_text_color_subdued_dark=TEXT_GRAY,
    background_fill_primary="#ffffff", background_fill_primary_dark="#ffffff",
    background_fill_secondary=CREAM, background_fill_secondary_dark=CREAM,
    border_color_primary=CARD_BORDER, border_color_primary_dark=CARD_BORDER,
    # blocks / panels
    block_background_fill="#ffffff", block_background_fill_dark="#ffffff",
    block_border_color=CARD_BORDER, block_border_color_dark=CARD_BORDER,
    block_label_background_fill="#ffffff", block_label_background_fill_dark="#ffffff",
    block_label_text_color=DEEP_GREEN, block_label_text_color_dark=DEEP_GREEN,
    block_title_text_color=DEEP_GREEN, block_title_text_color_dark=DEEP_GREEN,
    # inputs
    input_background_fill="#ffffff", input_background_fill_dark="#ffffff",
    input_border_color=CARD_BORDER, input_border_color_dark=CARD_BORDER,
    input_placeholder_color=TEXT_GRAY, input_placeholder_color_dark=TEXT_GRAY,
    # buttons
    button_primary_background_fill=DEEP_GREEN, button_primary_background_fill_dark=DEEP_GREEN,
    button_primary_text_color="#ffffff", button_primary_text_color_dark="#ffffff",
    button_secondary_background_fill="#ffffff", button_secondary_background_fill_dark="#ffffff",
    button_secondary_text_color=TEXT_DARK, button_secondary_text_color_dark=TEXT_DARK,
    button_secondary_border_color=CARD_BORDER, button_secondary_border_color_dark=CARD_BORDER,
    # chatbot bubble accents
    color_accent=USER_BUBBLE, color_accent_soft=USER_BUBBLE, color_accent_soft_dark=USER_BUBBLE,
)

# ---------------- Real photo watermark URLs (Wikimedia Commons) ----------------
SUPREME_COURT_IMG = "https://commons.wikimedia.org/wiki/Special:FilePath/Supreme%20Court%20of%20Pakistan%2C%20Islamabad%20by%20Usman%20Ghani.jpg"
SCALES_IMG = "https://commons.wikimedia.org/wiki/Special:FilePath/Scale%20of%20justice.svg"

# ---------------- CSS: layout + belt-and-braces color overrides ----------------
custom_css = f"""
html, body {{ background: {CREAM} !important; }}
.gradio-container {{
    background: {CREAM} !important;
    max-width: 1180px !important;
    margin: auto !important;
    padding: 0 !important;
    font-family: 'Segoe UI', system-ui, sans-serif !important;
    position: relative !important;
    overflow: hidden !important;
    color: {TEXT_DARK} !important;
    box-shadow: 0 0 40px rgba(0,0,0,0.08);
}}
footer {{visibility: hidden}}

/* ---- Top green border with flag ---- */
#top-bar {{
    height: 34px; width: 100%; background: {DEEP_GREEN};
    display: flex; align-items: center; padding: 0 18px;
}}
#top-bar .flag {{ font-size: 18px; line-height: 1; }}

/* ---- Header ---- */
#header-row {{
    display: flex; align-items: center; justify-content: space-between;
    padding: 14px 32px; background: {DEEP_GREEN} !important;
    position: relative; z-index: 3;
}}
#logo-block {{ display: flex; align-items: center; gap: 14px; }}
#crest-badge {{
    width: 44px; height: 44px; border-radius: 8px;
    background: #ffffff; display: flex; align-items: center; justify-content: center;
}}
#logo-text h1 {{ font-family: Georgia, serif; color: #ffffff !important; font-size: 22px; margin: 0; line-height: 1.2; }}
#logo-text p {{ color: #d7e6db !important; font-size: 13px; margin: 3px 0 0 0; }}

/* ---- Watermark: court LEFT / tarazu RIGHT ---- */
#photo-watermark {{ position: absolute; inset: 0; z-index: 0; pointer-events: none; overflow: hidden; }}
#photo-watermark .court-photo {{
    position: absolute; top: 0; left: -5%; width: 55%; height: 100%;
    background-image: url('{SUPREME_COURT_IMG}');
    background-size: cover; background-position: center;
    opacity: 0.10; filter: grayscale(55%) brightness(1.35) contrast(0.9);
    mix-blend-mode: multiply;
}}
#photo-watermark .scales-photo {{
    position: absolute; top: 8%; right: -2%; width: 26%; height: 46%;
    background-image: url('{SCALES_IMG}');
    background-size: contain; background-repeat: no-repeat; background-position: right top;
    opacity: 0.14; filter: grayscale(70%) brightness(1.2) sepia(0.25);
    mix-blend-mode: multiply;
}}
#photo-watermark .sheen {{
    position: absolute; inset: 0;
    background: linear-gradient(115deg, rgba(255,255,255,0) 30%, rgba(255,255,255,0.35) 48%, rgba(255,255,255,0) 62%);
    mix-blend-mode: soft-light;
}}

/* ---- Welcome / hero ---- */
#welcome {{ position: relative; z-index: 1; text-align: center; padding: 50px 30px 30px 30px !important; }}
.badge-wrap {{ display: flex; justify-content: center; margin-bottom: 10px; }}
.dots-row {{ text-align:center; color: {GOLD}; font-size: 13px; letter-spacing: 14px; margin: 8px 0 20px 2px; }}
#welcome h2 {{ font-family: Georgia, serif; font-weight: 700; color: {DEEP_GREEN} !important; font-size: 36px; margin: 0 0 12px 0; }}
#welcome p.sub {{ color: {TEXT_GRAY} !important; font-size: 15px; margin-bottom: 26px; }}

#cards-row {{ gap: 16px !important; margin-bottom: 22px !important; }}
.card-btn {{
    background: #ffffff !important; border: 1px solid {CARD_BORDER} !important;
    border-radius: 14px !important; text-align: left !important;
    padding: 20px 18px 40px 18px !important; font-size: 15px !important;
    color: {TEXT_DARK} !important; font-family: Georgia, serif !important;
    position: relative !important; min-height: 84px !important;
    box-shadow: 0 1px 3px rgba(0,0,0,0.04); transition: all .18s ease;
    white-space: pre-line !important; line-height: 1.35 !important;
}}
.card-btn:hover {{ border-color: {MID_GREEN} !important; box-shadow: 0 4px 12px rgba(13,61,43,0.14); transform: translateY(-2px); }}

/* ---- Chat panel ---- */
#chatbot {{
    position: relative; z-index: 1;
    border: 1px solid {CARD_BORDER} !important;
    background: #ffffff !important;
    border-radius: 14px !important;
    margin: 10px 20px 0 20px !important;
    min-height: 48vh !important;
}}
/* belt-and-braces: force readable text no matter the internal DOM version */
#chatbot, #chatbot * {{ color: {TEXT_DARK} !important; }}
#chatbot .message.user {{ background: {USER_BUBBLE} !important; border: 1px solid {CARD_BORDER} !important; border-radius: 16px !important; }}
#chatbot .message.bot {{ background: {BOT_BUBBLE} !important; border: 1px solid {CARD_BORDER} !important; border-radius: 16px !important; }}
#chatbot button {{ background: #ffffff !important; border: 1px solid {CARD_BORDER} !important; color: {TEXT_DARK} !important; }}

/* ---- Input bar ---- */
#input-wrap {{
    position: sticky; bottom: 0; z-index: 3;
    background: linear-gradient(180deg, rgba(247,243,234,0) 0%, {CREAM} 35%);
    padding: 12px 20px 6px 20px !important;
}}
#input-row {{
    background: #ffffff !important; border: 1px solid {CARD_BORDER};
    border-radius: 16px; padding: 6px !important; align-items: center !important;
    box-shadow: 0 4px 16px rgba(0,0,0,0.06);
}}
#msg-box, #msg-box textarea, #msg-box input {{ background: #ffffff !important; color: {TEXT_DARK} !important; }}
#msg-box textarea {{ border: none !important; box-shadow: none !important; font-size: 14.5px !important; padding-left: 10px !important; }}
#msg-box textarea::placeholder {{ color: {TEXT_GRAY} !important; }}
#send-btn {{
    background: {DEEP_GREEN} !important; border-radius: 11px !important;
    min-width: 48px !important; max-width: 48px !important; height: 46px !important;
    color: {GOLD} !important; font-size: 19px !important;
}}

/* ---- Footer ---- */
#disclaimer-wrap {{ text-align: center; padding-top: 4px !important; padding-bottom: 12px !important; }}
#disclaimer {{ color: {TEXT_GRAY} !important; font-size: 12.5px; margin: 0 !important; font-weight: 500; }}
#credit {{ color: #9a9a8c !important; font-size: 10.5px; margin: 2px 0 0 0 !important; letter-spacing: 0.3px; }}
#credit b {{ color: {MID_GREEN} !important; font-weight: 600; }}
"""

crest_icon_svg = """
<svg width="24" height="24" viewBox="0 0 24 24" fill="none">
  <path d="M12 2 L20 5 V11 C20 16 16.5 20 12 22 C7.5 20 4 16 4 11 V5 Z" fill="#f0e6d2" stroke="#b8935a" stroke-width="1"/>
  <circle cx="12" cy="10" r="3" fill="none" stroke="#0d3d2b" stroke-width="1.3"/>
  <polygon points="15,7 15.8,8.5 17.5,8.7 16.2,9.8 16.6,11.5 15,10.6 13.4,11.5 13.8,9.8 12.5,8.7 14.2,8.5" fill="#0d3d2b"/>
</svg>
"""

badge_svg = """
<svg width="130" height="130" viewBox="0 0 130 130">
  <circle cx="65" cy="65" r="62" fill="#ffffff" stroke="#b8935a" stroke-width="2.5"/>
  <circle cx="65" cy="65" r="52" fill="none" stroke="#b8935a" stroke-width="1" stroke-dasharray="2,3" opacity="0.6"/>
  <g fill="#b8935a">
    <path d="M65 18 a10 10 0 1 0 0.1 0 a8 8 0 1 1 -0.1 0" />
    <polygon points="72,16 73.5,19 76.5,19.2 74,21 74.8,24 72,22.3 69.2,24 70,21 67.5,19.2 70.5,19"/>
  </g>
  <g fill="none" stroke="#b8935a" stroke-width="2.4" stroke-linecap="round">
    <line x1="65" y1="40" x2="65" y2="90"/>
    <line x1="42" y1="52" x2="88" y2="52"/>
    <line x1="42" y1="52" x2="35" y2="72"/>
    <line x1="42" y1="52" x2="49" y2="72"/>
    <path d="M32 72 a10 8 0 0 0 20 0"/>
    <line x1="88" y1="52" x2="81" y2="72"/>
    <line x1="88" y1="52" x2="95" y2="72"/>
    <path d="M78 72 a10 8 0 0 0 20 0"/>
    <line x1="52" y1="92" x2="78" y2="92"/>
    <line x1="65" y1="88" x2="65" y2="94"/>
  </g>
</svg>
"""

# ---------------- Assemble app ----------------
with gr.Blocks(theme=theme, css=custom_css, title="Pakistan Constitution AI Assistant") as demo:

    gr.HTML("""
        <div id="photo-watermark">
            <div class="court-photo"></div>
            <div class="scales-photo"></div>
            <div class="sheen"></div>
        </div>
    """)

    with gr.Row(elem_id="header-row"):
        gr.HTML(f"""
            <div id="logo-block">
                <div id="crest-badge">{crest_icon_svg}</div>
                <div id="logo-text">
                    <h1>Pakistan Constitution AI Assistant</h1>
                    <p>Your intelligent legal research companion</p>
                </div>
            </div>
        """)

    with gr.Column(elem_id="welcome", visible=True) as welcome_col:
        gr.HTML(f"""
            <div class="badge-wrap">{badge_svg}</div>
            <div class="dots-row">✦&nbsp;&nbsp;&nbsp;✦</div>
            <h2>How can I help you today?</h2>
            <p class="sub">Ask about any article, discuss a legal scenario, or understand constitutional rights.</p>
        """)
        with gr.Row(elem_id="cards-row"):
            card1 = gr.Button("What is Article 19\nabout?", elem_classes="card-btn")
            card2 = gr.Button("Explain fundamental\nrights", elem_classes="card-btn")
            card3 = gr.Button("A scenario: illegal\narrest", elem_classes="card-btn")
            card4 = gr.Button("Difference between\nArticle 8 and 25", elem_classes="card-btn")

    chatbot = gr.Chatbot(elem_id="chatbot", visible=False,
                          label="⚖️  Legal Assistant", show_label=True, height=430)

    with gr.Column(elem_id="input-wrap"):
        with gr.Row(elem_id="input-row"):
            msg = gr.Textbox(
                placeholder="Type your legal question or scenario...",
                show_label=False, scale=9, elem_id="msg-box", lines=1
            )
            send_btn = gr.Button("➤", elem_id="send-btn", scale=1)
        with gr.Column(elem_id="disclaimer-wrap"):

            gr.HTML("<p id='credit'>Smart AI Legal Assistant By <b>Hamza</b></p>")

    send_btn.click(run_rag, [msg, chatbot], [chatbot, msg, welcome_col, chatbot])
    msg.submit(run_rag, [msg, chatbot], [chatbot, msg, welcome_col, chatbot])

    card_questions = {
        card1: "What is Article 19 about?",
        card2: "Explain fundamental rights",
        card3: "A scenario: illegal arrest",
        card4: "Difference between Article 8 and 25",
    }
    for btn, q in card_questions.items():
        btn.click(lambda q=q: q, None, msg).then(
            run_rag, [msg, chatbot], [chatbot, msg, welcome_col, chatbot]
        )

demo.launch(share=True, debug=True)

/tmp/ipykernel_4284/2112264175.py:236: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=custom_css, title="Pakistan Constitution AI Assistant") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e41ce95d27d67a23d1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d6615a9c95a0a43a25.gradio.live
Killing tunnel 127.0.0.1:7860 <> https://e41ce95d27d67a23d1.gradio.live
